# Modelo Relacional — dim_host_sk + fact_planet_sk
**Pipeline stage:** `DIM/FACT`

Construimos el modelo estrella (star schema) según el Data Contract:

| Tabla | Grain | PK | 
|---|---|---|
| `dim_host_sk` | 1 fila por `hostname` | `host_id` (surrogate) — UNIQUE: `hostname` |
| `fact_planet_sk` | 1 fila por `pl_name` | `pl_name` — FK: `host_id → dim_host_sk.host_id` |


In [1]:
import pandas as pd, numpy as np
import duckdb, pyarrow as pa, pyarrow.parquet as pq
import os

SILVER_PATH = "../data/silver/silver_planet.parquet"
df = pd.read_parquet(SILVER_PATH)
print(f"Silver cargado: {df.shape}")

Silver cargado: (6123, 22)


## dim_host_sk — 1 fila por hostname (estrella)

In [2]:
# Columnas de la dimensión (propiedades de la estrella anfitriona)
DIM_COLS = [
    "hostname", "n_stars", "n_planets", "dist_pc",
    "st_spectype", "spectral_class", "hr_region",
    "st_teff_k", "st_rad_rsun", "st_mass_msun",
    "st_lum_log", "st_logg", "st_met_dex",
]

# Un hostname puede aparecer en múltiples planetas -> tomar la mediana de métricas y la moda para campos categóricos
dim_num  = df.groupby("hostname")[
    [c for c in DIM_COLS if c not in ("hostname","st_spectype","spectral_class","hr_region","n_stars","n_planets")]
].median().reset_index()

dim_cat  = df.groupby("hostname")[["st_spectype","spectral_class","hr_region","n_stars","n_planets"]].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else pd.NA
).reset_index()

dim_host = dim_num.merge(dim_cat, on="hostname", how="left")

# Surrogate key: host_id (secuencial, ordenado por hostname)
dim_host = dim_host.sort_values("hostname").reset_index(drop=True)
dim_host.insert(0, "host_id", dim_host.index + 1)

print(f"dim_host_sk shape: {dim_host.shape}")
print(f"Hostnames únicos : {dim_host['hostname'].nunique():,}")
print(f"Duplicados en hostname: {dim_host['hostname'].duplicated().sum()}  <- debe ser 0")
dim_host.head(3)

dim_host_sk shape: (4557, 14)
Hostnames únicos : 4,557
Duplicados en hostname: 0  <- debe ser 0


,host_id,hostname,dist_pc,st_teff_k,st_rad_rsun,st_mass_msun,st_lum_log,st_logg,st_met_dex,st_spectype,spectral_class,hr_region,n_stars,n_planets
0,1,11 Com,93.1846,4874.0,13.76,2.09,1.97823,2.45,-0.26,G8 III,G,Supergiant,2,1
1,2,11 UMi,125.3210,4213.0,29.79,2.78,2.42951,1.93,-0.02,K4 III,K,Supergiant,1,1
2,3,14 And,75.4392,4888.0,11.55,1.78,1.83992,2.55,-0.21,K0 III,K,Giant,1,1


## fact_planet_sk — 1 fila por pl_name (planeta)

In [4]:
# Columnas del hecho (propiedades del planeta + FK a estrella)
FACT_COLS = [
    "pl_name", "hostname",
    "pl_orbper_d", "pl_rad_rearth", "pl_mass_mearth", "pl_eqt_k",
    "pl_size_class", "hab_zone_flag",
    "disc_year", "disc_method",
]

fact_planet = df[FACT_COLS].drop_duplicates(subset="pl_name").copy()

# Unir FK host_id desde dim_host
fact_planet = fact_planet.merge(
    dim_host[["host_id", "hostname"]], on="hostname", how="left"
)

# Reordenar: pl_name primero, luego FK, luego métricas
fact_planet = fact_planet[
    ["pl_name", "host_id", "hostname",
     "pl_orbper_d", "pl_rad_rearth", "pl_mass_mearth", "pl_eqt_k",
     "pl_size_class", "hab_zone_flag", "disc_year", "disc_method"]
].copy()

print(f"fact_planet_sk shape: {fact_planet.shape}")
print(f"Planetas únicos (pl_name) : {fact_planet['pl_name'].nunique():,}")
print(f"Duplicados en pl_name     : {fact_planet['pl_name'].duplicated().sum()}  <- debe ser 0")
fact_planet.head(3)

fact_planet_sk shape: (6123, 11)
Planetas únicos (pl_name) : 6,123
Duplicados en pl_name     : 0  <- debe ser 0


,pl_name,host_id,hostname,pl_orbper_d,pl_rad_rearth,pl_mass_mearth,pl_eqt_k,pl_size_class,hab_zone_flag,disc_year,disc_method
0,Kepler-1167 b,1764,Kepler-1167,1.003934,1.710000,3.570,1419.0,Super-Earth,0,2016,Transit
1,Kepler-1740 b,2392,Kepler-1740,8.172400,3.323214,11.000,858.0,Sub-Neptune,0,2021,Transit
2,Kepler-1581 b,2223,Kepler-1581,6.283855,0.800000,0.437,1108.0,Rocky,0,2016,Transit


## Checks de integridad referencial (evidencia)

In [6]:
print("=" * 55)
print("CHECK 1 Unicidad PK dim — COUNT(*) == COUNT(DISTINCT hostname)")
n_dim = len(dim_host)
n_uniq = dim_host["hostname"].nunique()
check1 = "PASS" if n_dim == n_uniq else "✗ FAIL"
print(f"n_rows(dim) = {n_dim:,}  |  n_keys(hostname) = {n_uniq:,} -> {check1}")

print("\nCHECK 2 Unicidad PK fact — pl_name único")
n_fact = len(fact_planet)
n_uniq_pl = fact_planet["pl_name"].nunique()
check2 = "PASS" if n_fact == n_uniq_pl else "FAIL"
print(f"n_rows(fact) = {n_fact:,}  |  n_keys(pl_name) = {n_uniq_pl:,} -> {check2}")

print("\nCHECK 3 Orphan rows — fact.host_id sin match en dim")
orphan_rows = fact_planet["host_id"].isna().sum()
check3 = "PASS" if orphan_rows == 0 else f"FAIL ({orphan_rows} orphans)"
print(f"orphan_rows = {orphan_rows} -> {check3}")

print("\nCHECK 4 FK referencial: todo host_id en fact está en dim")
fact_ids = set(fact_planet["host_id"].dropna().astype(int))
dim_ids = set(dim_host["host_id"].astype(int))
dangling = fact_ids - dim_ids
check4 = "PASS" if len(dangling) == 0 else f"FAIL ({len(dangling)} dangling FKs)"
print(f"dangling FKs = {len(dangling)} -> {check4}")

print("\n" + "=" * 55)
all_pass = all(x.startswith("PASS") for x in [check1, check2, check3, check4])
print("RESULTADO FINAL:", "ALL CHECKS PASSED" if all_pass else "HAY FALLOS")

CHECK 1 Unicidad PK dim — COUNT(*) == COUNT(DISTINCT hostname)
n_rows(dim) = 4,557  |  n_keys(hostname) = 4,557 -> PASS

CHECK 2 Unicidad PK fact — pl_name único
n_rows(fact) = 6,123  |  n_keys(pl_name) = 6,123 -> PASS

CHECK 3 Orphan rows — fact.host_id sin match en dim
orphan_rows = 0 -> PASS

CHECK 4 FK referencial: todo host_id en fact está en dim
dangling FKs = 0 -> PASS

RESULTADO FINAL: ALL CHECKS PASSED


## Guardar dim y fact

In [7]:
os.makedirs("../data/gold", exist_ok=True)
os.makedirs("../artifacts", exist_ok=True)

# Parquet (pipeline interno)
pq.write_table(pa.Table.from_pandas(dim_host,   preserve_index=False),
               "../data/gold/dim_host_sk.parquet", compression="snappy")
pq.write_table(pa.Table.from_pandas(fact_planet, preserve_index=False),
               "../data/gold/fact_planet_sk.parquet", compression="snappy")

# CSV evidencia → artifacts/
dim_host.to_csv("../artifacts/dim_host_sk.csv", index=False)
fact_planet.to_csv("../artifacts/fact_planet_sk.csv", index=False)

print("dim_host_sk.parquet + .csv  guardados")
print("fact_planet_sk.parquet + .csv guardados")
print(f"dim  : {len(dim_host):,} filas x {dim_host.shape[1]} cols")
print(f"fact : {len(fact_planet):,} filas x {fact_planet.shape[1]} cols")

dim_host_sk.parquet + .csv  guardados
fact_planet_sk.parquet + .csv guardados
dim  : 4,557 filas x 14 cols
fact : 6,123 filas x 11 cols


## Vista DuckDB del modelo

In [8]:
con = duckdb.connect()
con.execute(f"CREATE VIEW dim  AS SELECT * FROM read_parquet('../data/gold/dim_host_sk.parquet')")
con.execute(f"CREATE VIEW fact AS SELECT * FROM read_parquet('../data/gold/fact_planet_sk.parquet')")

result = con.execute("""
    SELECT
        d.spectral_class,
        COUNT(DISTINCT d.host_id) AS n_stars,
        COUNT(f.pl_name) AS n_planets,
        ROUND(AVG(d.st_teff_k), 0) AS avg_teff_k,
        ROUND(AVG(d.st_mass_msun),2) AS avg_mass_msun,
        ROUND(AVG(f.pl_rad_rearth),2) AS avg_pl_rad
    FROM dim d
    JOIN fact f ON d.host_id = f.host_id
    GROUP BY d.spectral_class
    ORDER BY avg_teff_k DESC
""").df()
print(result.to_string())

  spectral_class  n_stars  n_planets  avg_teff_k  avg_mass_msun  avg_pl_rad
0              B        7          8     15990.0           2.98       10.60
1              A       20         24      7747.0           1.76       15.78
2              F      236        286      6261.0           1.29       12.43
3              G      564        780      5627.0           1.07        9.49
4        Unknown     2949       3836      5574.0           0.93        4.29
5              K      436        646      4840.0           1.04        8.53
6              L        1          1      4034.0           0.08       11.90
7              M      344        542      3635.0           0.44        3.88


## Checkpoint DIM/FACT
| Check | Estado |
|---|---|
| `dim_host_sk` — 1 fila por hostname | ✓ |
| `fact_planet_sk` — 1 fila por pl_name | ✓ |
| PK uniqueness (dim y fact) | ✓ |
| `orphan_rows == 0` | ✓ |
| FK `fact.host_id → dim.host_id` | ✓ |
| Artefactos CSV generados | ✓ |